# 00 — Chạy toàn bộ dự án FPT

Dùng bản này nếu muốn chạy một lần từ dữ liệu đến mô hình và dự báo. Không cần chạy thêm cả 7 notebook riêng. Những notebook đó dùng để học sâu từng bước.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Bản clone đầy đủ; thư viện trong requirements-notebooks.txt (local).

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Kiểm định dữ liệu giá

Snapshot cố định có 4.671 phiên. Ô này chỉ đọc/kiểm tra, không thay dữ liệu gốc và không gọi API tài chính.

In [ ]:
from fetch_data import verify_snapshot
verification = verify_snapshot(PRICE_PATH, expected_last_date=CUTOFF)
display(pd.Series(verification, name="Giá trị kiểm định").to_frame())
prices = pd.read_csv(PRICE_PATH, parse_dates=["time"])
display(prices.head(3))
display(prices.tail(3))

## 5. Chuẩn hóa và ghép dữ liệu bổ trợ

Dùng các CSV nguồn đã lưu. Ghép theo ngày có thể biết, không điền ngược. Độ trễ báo cáo quý/GDP là giả định thực nghiệm, chưa phải kho lịch sử từng phiên bản.

In [ ]:
from fetch_supplemental import run_from_cache, MODEL_SUPPLEMENTAL_COLUMNS
# Chuẩn hóa CSV đã đóng gói; không tải mạng và không nội suy từ tương lai.
supplement_audit = run_from_cache(RAW_SUPPLEMENTAL_DIR, SUPPLEMENTAL_PATH, end=CUTOFF)
supplement = pd.read_csv(SUPPLEMENTAL_PATH, parse_dates=["date"])
display(pd.DataFrame(supplement_audit["source_audit"]).T)
print("Đã ghi:", SUPPLEMENTAL_PATH.relative_to(ROOT))

## 6. Khám phá và kiểm toán dữ liệu

Kiểm tra khoảng ngày, nhãn, khối lượng bằng 0 và OHLC bất nhất; lưu JSON và các biểu đồ.

In [ ]:
from eda import run_eda
audit = run_with_log(run_eda, PRICE_PATH, FIGURES_DIR, REPORTS_DIR)
display(pd.Series({key: audit[key] for key in [
    "row_count", "date_start", "date_end", "duplicate_dates", "zero_volume_sessions", "ohlc_consistency_violations"
]}).to_frame("Kiểm toán"))
display(pd.Series(audit["next_session_direction_counts"]).to_frame("Số phiên"))

## 7. Xem biểu đồ EDA

Mỗi ảnh được tạo bằng số liệu vừa chạy; ảnh đồng thời nằm trong figures/.

In [ ]:
for name in ["01_fpt_close.png", "02_fpt_volume.png", "03_daily_return_distribution.png", "04_next_session_class_balance.png"]:
    show_figure(name)

## 8. Tạo đầu vào và nhãn

Nhánh 1 có 32 biến, Nhánh 2 có 44 biến. Hai nhánh dùng cùng những ngày có đủ dữ liệu. Nhãn t+1 được giữ riêng, không đưa vào đầu vào.

In [ ]:
from prepare import prepare, build_experiment_table, BRANCH_PRICE, BRANCH_MACRO
if not SUPPLEMENTAL_PATH.is_file():
    raise FileNotFoundError("Chạy notebook 02 hoặc bước dữ liệu bổ trợ trong 00_RUN_ALL trước.")
feature_rows, labeled_rows = prepare(PRICE_PATH, PROCESSED_DIR / "fpt_next_session_features.csv", SUPPLEMENTAL_PATH)
base, common, branches = build_experiment_table(PRICE_PATH, SUPPLEMENTAL_PATH)
display(pd.DataFrame([{"Nhánh": name, "Số biến": len(columns)} for name, columns in branches.items()]))
display(common[["date", "target_date", "current_close", "next_close", "direction"]].tail())

## 9. Kiểm tra Train 85% / Test 15%

Chia theo ngày nhãn; không xáo trộn. Bỏ phiên đứng giá khỏi phân loại sau khi đã xác định mốc chung.

In [ ]:
from train import _chronological_splits
masks, boundary = _chronological_splits(common)
display(pd.Series(boundary).to_frame("Train/Test"))
partition_rows = []
for part, mask in masks.items():
    subset = common.loc[mask]
    partition_rows.append({
        "Tập": part, "Mẫu hồi quy": len(subset),
        "Mẫu phân loại": int(subset["direction"].notna().sum()),
        "Ngày nhãn đầu": subset["target_date"].min(),
        "Ngày nhãn cuối": subset["target_date"].max(),
    })
display(pd.DataFrame(partition_rows))
assert common.loc[masks["train"], "target_date"].max() < common.loc[masks["test"], "target_date"].min()

## 10. Huấn luyện và xuất toàn bộ kết quả

Chạy đủ 5 phương pháp mỗi bài toán trên mỗi nhánh (20 cấu hình kể cả baseline). StandardScaler nằm trong Pipeline và chỉ fit Train khi chấm Test.

In [ ]:
from train import run_training
# Cùng hàm dùng ở CLI: fit Train, chấm Test, xuất bảng/ảnh, rồi refit mô hình được chọn.
# CPU đủ cho dữ liệu này. Chờ ô hoàn tất, không bấm chạy trùng lần hai.
training_summary = run_with_log(
    run_training, PRICE_PATH, FIGURES_DIR, REPORTS_DIR, MODELS_DIR, PROCESSED_DIR, SUPPLEMENTAL_PATH
)
display(pd.DataFrame({
    "Phân loại": training_summary["classifier_selected"],
    "Hồi quy": training_summary["regressor_selected"],
}))

## 11. Đọc kết quả phân loại

Accuracy là tỷ lệ đúng hướng. Balanced Accuracy là trung bình tỷ lệ nhận đúng Tăng và Giảm. F1 cân bằng Precision/Recall của lớp Tăng; ROC-AUC chấm cách xếp hạng xác suất.

In [ ]:
classification = pd.read_csv(REPORTS_DIR / "classification_test_metrics.csv")
display(classification[["feature_set", "model", "test_rows", "accuracy", "balanced_accuracy", "precision_up", "recall_up", "f1_up", "roc_auc"]].round(4))
# Các tỷ lệ trên nằm trong [0, 1]; nhân 100 nếu muốn đọc theo phần trăm.
display(pd.read_csv(REPORTS_DIR / "confusion_matrices.csv"))

## 12. Đọc sai số dự đoán giá

MAE/RMSE tính bằng nghìn VND/cổ phiếu; MAPE và các cột *_percent đã là %. Tolerance là tỷ lệ dự đoán trong ngưỡng. Last-Close giữ nguyên giá nên không đoán đúng hướng ở phiên thật sự tăng/giảm.

In [ ]:
regression = pd.read_csv(REPORTS_DIR / "regression_test_metrics.csv")
display(regression[["feature_set", "model", "mae", "rmse", "mape_percent", "directional_accuracy_nonflat_percent",
                    "tolerance_accuracy_0_5_percent", "tolerance_accuracy_1_percent", "tolerance_accuracy_2_percent"]].round(4))
display(pd.read_csv(REPORTS_DIR / "ablation_summary.csv"))

## 13. Xem đường dự báo và đối sánh

Đường đen là thực tế, các đường còn lại là dự báo từ mô hình chỉ học Train. Phóng to 80 phiên để thấy sai lệch gần hơn.

In [ ]:
for name in ["07_regression_lines_by_branch.png", "08_branch_xgboost_comparison_zoom.png", "09_classification_metric_bars.png", "10_regression_metric_bars.png"]:
    show_figure(name)

## 14. Dự báo sau phiên cuối snapshot

Mô hình được chọn sau đối chiếu Test rồi học lại toàn bộ nhãn đã biết. Việc chọn trên Test làm điểm của mô hình được chọn mang tính thăm dò. Dự báo này là minh họa sau 30/09/2025, không phải dự báo theo giá thị trường hôm nay.

In [ ]:
from predict import predict_latest
needed_models = [MODELS_DIR / "best_classifier.joblib", MODELS_DIR / "best_regressor.joblib"]
if not all(path.is_file() for path in needed_models):
    raise FileNotFoundError("Chưa có mô hình: chạy bước train ở 00_RUN_ALL hoặc notebook 06 trước.")
prediction = run_with_log(predict_latest, PRICE_PATH, MODELS_DIR, REPORTS_DIR / "latest_prediction.json", SUPPLEMENTAL_PATH)
display(pd.Series(prediction).to_frame("Dự báo"))

## 15. Chạy bộ kiểm thử

Chạy pytest bằng đúng Python của kernel; kiểm thử phát hiện lỗi sẽ làm ô dừng và hiện thông báo. Không dùng giá trị nhãn tương lai để tạo đầu vào.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, check=True)